# PyTorch Demo

<div align=right>Author: Jiachen Li</div>
<div align=right>Artificial Intelligence<br>2021 Spring<br>ISEE, Zhejiang University</div>

In this demo, we are going to learn how to bulid a simple fully-connected neural network with deep learning framework PyTorch. The goal is to run a classification task on handwritten digits. Before this tutorial, please ensure that you have already successfully install all the requirements and correctly configurate the running environment.

**Tips:** If you have installed jupyter, you can directly run the codes with jupyter notebook.

## Prerequisites

In this demo, we use following dependencies:
- torch==1.6.0
- numpy
- tqdm

Check your torch installation with such python command:

```python
python
>>> import torch
>>> torch.__verion__
'1.6.0'
```

For those who use Nvidia GPUs, check if your CUDA configurations are correct to support current version of PyTorch.

### Dataset

In this demo, we use the well-known handwritten digit dataset MNIST. MNIST contains huge amount of handwritten digit images (grayscale) collected from real world. Each sample is a $28\times28$ pixel, single-channel grayscale digit, ranging from 0-9. We provide the dataset as an independent file `mnist.npz` with this demo.

In the training stage, images are fed into the model to fit optimal parameters, classifying the input image to its corresponding class correctly. In the inference stage, unused new images are fed into the model to compute the most possible class of the input, as the classification result.

Samples of MNIST dataset:

![sample1](./imgs/sample1.png)
![sample2](./imgs/sample2.png)
![sample3](./imgs/sample3.png)

Official website of MNIST dataset: http://yann.lecun.com/exdb/mnist/

## Quick Start

To use packages, you should import them before your use.

In [1]:
import torch
import numpy as np
import argparse, os, tqdm

### Initialization

Before all core codes, we need `argparse` to parse the command line arguments. In this demo, we pass a `--gpu` argument to assign GPU resource (if you have) for following computation. `argparser` is a simple but useful tool in many applications, check detail usage of it from its official document.

**NOTE:** If you run the code here, the following parser block should not be run, since in the interactive python envrionment there are no CMD arguments.

In [ ]:
# Argument parser
# Don't run in the interactive envrionment!
parser = argparse.ArgumentParser()
parser.add_argument('--gpu', default='', type=str, help='GPU IDs. E.g. 0,1,2. Do not add this argument if you are using CPU mode.')
args = parser.parse_args()
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu # Assign GPU with given the given argument

If you want to test its function, close the jupyter notebook and run it from CMD. For example:

```bash
python torch_demo.py --gpu 0
```

Here we just assign the GPU manually. If you don't use GPU, just leave it with the empty string.

In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '' # If you have GPU, fill the empty string with IDs, e.g. '0' or '0,1,2'

### Build Network

In PyTorch, a network model can be implemented via create a class inherited from `torch.nn.Module`. In this demo, we are going to build a simple fully-connected neural network with four FC layers. Between two layers, we use ReLU as the non-linear activation function. There are three hidden layers in total, with outputs with the shape of `emb_size`. The last layer has `num_class`-dim outputs, meaning the number of classes.

After the last layer, we apply a softmax function to convert the outputs into probabilities of each digit from 0-9. In actual code, the softmax function is not implemented in the model but applied together with the loss function. You will see it later.

In [3]:
# Simple demo network
class SimpleNN(torch.nn.Module):
    '''A simple fully-connected neural network demo.'''
    def __init__(self, input_size, emb_size, num_class):
        super(SimpleNN, self).__init__()
        self.input_size = input_size
        self.emb_size = emb_size
        self.num_class = num_class

        # FC layers
        self.fc1 = torch.nn.Linear(self.input_size, self.emb_size)
        self.fc2 = torch.nn.Linear(self.emb_size, self.emb_size)
        self.fc3 = torch.nn.Linear(self.emb_size, self.emb_size)
        self.fc4 = torch.nn.Linear(self.emb_size, self.num_class)

        # Activation layers
        self.activate = torch.nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.activate(x)
        x = self.fc2(x)
        x = self.activate(x)
        x = self.fc3(x)
        x = self.activate(x)
        x = self.fc4(x)
        return x

In `__init__`, we define the main parameters and sub-modules of the model. In `forward`, we define the dataflow pipeline.

### Build Dataset

In PyTorch, a dataset can be abstracted as a unique object, including data organizing, fetching and preprocessing. You should inherite the `torch.utils.data.Dataset` class to create your own dataset.

There are two crucial methods should be override. `__getitem__` define the behavior when fetching a sample from the dataset. It usually contains file reading operations and preprocessing works. `__len__` define the size of the dataset.

In [4]:
# Pytorch dataset definition
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, npz_file, mode='train'):
        assert mode == 'train' or mode == 'test' or mode == 'val', ('Mode should be [train|test|val]!')
        self.mode = mode
        data = np.load(npz_file)
        if mode == 'train':
            self.images = data['x_train'][:len(data['x_train'])//2]
            self.labels = data['y_train'][:len(data['y_train'])//2]
        elif mode == 'test':
            self.images = data['x_test']
            self.labels = data['y_test']
        elif mode == 'val':
            self.images = data['x_train'][len(data['x_train'])//2:]
            self.labels = data['y_train'][len(data['y_train'])//2:]

    def __getitem__(self, index):
        # Get current data
        img = self.images[index,:]
        label = self.labels[index]
        
        # Preprocessing
        img = torch.from_numpy(img).float()
        label = torch.as_tensor(label, dtype=torch.long).item()
        h, w = img.size(0), img.size(1)
        img = img / 255.0                  # [0,255] -> [0,1]
        img = img.view(h*w)              # Squeeze into a 1-dim vector
        img = img - torch.mean(img) # [0,1] -> [-1,1]
            
        return img, label

    def __len__(self):
        return self.labels.shape[0]

### Pipeline

Now all the preparations are completed. Let's start the main pipeline of model training.

At the very beginning, create dataset objects for training, testing and validating data.

In [6]:
# Prepare data
data_path = './mnist.npz'
train_dataset = SimpleDataset(data_path, mode='train')
test_dataset = SimpleDataset(data_path, mode='test')
val_dataset = SimpleDataset(data_path, mode='val')

To implement mini-batch iteration on the whole dataset, we need to create a special object `DataLoader` in PyTorch. The dataloader defines how to sample samples from the dataset (e.g. sampling strategy, batch size, etc.). Here we create three dataloaders for three dataset.

In [10]:
# Create dataloader
bsize = 16
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=bsize, shuffle=True, num_workers=8, drop_last=True)
test_loader = torch.utils.data.DataLoader(train_dataset, batch_size=bsize, shuffle=False, num_workers=8, drop_last=True)
val_loader = torch.utils.data.DataLoader(train_dataset, batch_size=bsize, shuffle=False, num_workers=8, drop_last=True)

Then we create the model with `input_size=28*28`, `emb_size=128` and `num_class=10`. Here we stretch each image into a 28\*28-dim vector to fit with the input size of the first layer of the model. You can check the stretching process in dataset definitions.

To update the parameters of the model, we use SGD (Stochastic Gradient Descent) optimizer. This optimizer computes gradient and update the parameters each batch, with a given hyper-parameter `lr` meaning the learning rate. Here we set `lr=0.01`.

In [11]:
# Create model
model = SimpleNN(input_size=28*28, emb_size=128, num_class=10)
if torch.cuda.is_available():
    model = model.cuda()

# Set optimizer
opt = torch.optim.SGD(model.parameters(), lr=0.01)

Finally, we complete the training and testing process. Check the codes with PyTorch docmument, try to figure out how to compute loss, gradient and accuracy. And you will see the training process messages on the CMD.

In [ ]:
# Training
epochs = 10
for epoch in range(epochs):
    avg_loss = 0
    cnt = 0
    for i, (img, lbl) in enumerate(tqdm.tqdm(train_loader)):
        # Clear optimizer history info.
        opt.zero_grad()

        if torch.cuda.is_available():
            img = img.cuda()
            lbl = lbl.cuda()
        pred = model(img)

        # Loss function
        loss = torch.nn.functional.cross_entropy(pred, lbl)
        avg_loss += loss.item()
        cnt += 1

        # Get gradients
        loss.backward()

        # Update model
        opt.step()
    avg_loss /= cnt

    # Validation
    # torch.no_grad() does not record gradients, designed for inference.
    avg_acc = 0
    cnt = 0
    with torch.no_grad():
        model = model.eval() # Set model in evaluation mode.
        for i, (img, lbl) in enumerate(val_loader):
            if torch.cuda.is_available():
                img = img.cuda()
                lbl = lbl.cuda()
            pred = model(img)
            pred = torch.argmax(pred, dim=1)
            nxor = ~torch.logical_xor(pred, lbl)
            n_correct = len(list(filter(lambda x: x == True, nxor)))
            avg_acc += n_correct / len(nxor)
            cnt += 1
        model = model.train() # Return to training mode.
    avg_acc /= cnt
    print('-----------------------------------------------------------------')
    print('Epoch:{}, avg_loss:{:.4f}, avg_val_acc:{:.2f}%'.format(epoch, avg_loss, avg_acc*100))
    print('-----------------------------------------------------------------')
print('>>> Training finished. Start evaluation...')

# Evaluation
avg_acc = 0
cnt = 0
with torch.no_grad():
    model = model.eval() # Set model in evaluation mode.
    for i, (img, lbl) in enumerate(test_loader):
        if torch.cuda.is_available():
            img = img.cuda()
            lbl = lbl.cuda()
        pred = model(img)
        pred = torch.argmax(pred, dim=1)
        nxor = ~torch.logical_xor(pred, lbl)
        n_correct = len(list(filter(lambda x: x == True, nxor)))
        avg_acc += n_correct / len(nxor)
        cnt += 1
    model = model.train() # Return to training mode.
avg_acc /= cnt
print('Evaluation result:')
print('Acc.:{:.2f}'.format(avg_acc*100))

  0%|          | 0/1875 [00:00<?, ?it/s]

-----------------------------------------------------------------
Epoch:0, avg_loss:1.7082, avg_val_acc:98.65%
-----------------------------------------------------------------


  0%|          | 0/1875 [00:00<?, ?it/s]

-----------------------------------------------------------------
Epoch:1, avg_loss:0.4394, avg_val_acc:98.72%
-----------------------------------------------------------------


  0%|          | 0/1875 [00:00<?, ?it/s]

...

  0%|          | 0/1875 [00:00<?, ?it/s]

-----------------------------------------------------------------
Epoch:34, avg_loss:0.0036, avg_val_acc:100.00%
-----------------------------------------------------------------


  0%|          | 0/1875 [00:00<?, ?it/s]

-----------------------------------------------------------------
Epoch:35, avg_loss:0.0032, avg_val_acc:100.00%
-----------------------------------------------------------------


 80%|████████  | 1506/1875 [00:04<00:00, 384.16it/s]